In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw'

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 192) / Test: (90067, 191)


In [2]:
# feature_refinement 재적용
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)
    # 시술 시기 코드 관련 피처 제외 (랜덤 코드 확인)

print('피처 정제 완료')

피처 정제 완료


In [3]:
# 피처 준비
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
X_xgb[cat_features]      = enc.fit_transform(X[cat_features])
X_test_xgb[cat_features] = enc.transform(X_test[cat_features])
num_features = [c for c in feat_cols if c not in cat_features]
medians = X_xgb[num_features].median()
X_xgb[num_features]      = X_xgb[num_features].fillna(medians)
X_test_xgb[num_features] = X_test_xgb[num_features].fillna(medians)

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

N_SPLITS = 5
print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

피처: 197개 / 범주형: 21개
scale_pos_weight: 2.87


In [ ]:
# Seed Averaging
# 각 SEED마다 3모델 5-fold 학습 -> OOF/test 에측값 누적-> 마지막에 평균
SEEDS = [42, 0, 123, 2024, 777]

# 전체 SEED에 걸쳐 누적할 배열
oof_cat_all  = np.zeros(len(X))
oof_lgb_all  = np.zeros(len(X))
oof_xgb_all  = np.zeros(len(X))
test_cat_all = np.zeros(len(X_test))
test_lgb_all = np.zeros(len(X_test))
test_xgb_all = np.zeros(len(X_test))

seed_results = []

for seed_idx, SEED in enumerate(SEEDS):
    print('\n' + '='*50)    
    print(f'SEED {SEED} ({seed_idx+1}/{len(SEEDS)})')
    print('\n' + '='*50)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    cat_params = {
        'iterations': 794, 'learning_rate': 0.030094391673729362,
        'depth': 7, 'l2_leaf_reg': 7.433949857398995,
        'bagging_temperature': 0.3980358270898108,
        'random_strength': 1.3075975210328405, 'border_count': 108,
        'loss_function': 'Logloss', 'eval_metric': 'AUC',
        'scale_pos_weight': spw, 'random_seed': SEED,
        'early_stopping_rounds': 50, 'verbose': False, 'use_best_model': True,
    }
    lgb_params = {
        'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
        'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.02030926523583015,
        'n_estimators': 956, 'subsample': 0.9916893928795039,
        'colsample_bytree': 0.8760988294276582,
        'reg_alpha': 0.5158539052029842, 'reg_lambda': 0.05557192411355649,
        'min_child_samples': 98,
        'scale_pos_weight': spw, 'random_state': SEED, 'verbose': -1,
    }
    xgb_params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
        'max_depth': 4, 'learning_rate': 0.028817004923326044,
        'n_estimators': 835, 'subsample': 0.8755018207309047,
        'colsample_bytree': 0.8959610275285067,
        'reg_alpha': 0.9163081802804784, 'reg_lambda': 1.1472566862699578,
        'min_child_weight': 8, 'gamma': 0.728514692891076,
        'scale_pos_weight': spw, 'random_state': SEED,
        'verbosity': 0, 'early_stopping_rounds': 50,
    }

    oof_cat  = np.zeros(len(X))
    oof_lgb  = np.zeros(len(X))
    oof_xgb  = np.zeros(len(X))
    test_cat = np.zeros(len(X_test))
    test_lgb = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr  = X.iloc[tr_idx].reset_index(drop=True)
        X_val = X.iloc[val_idx].reset_index(drop=True)
        y_tr  = y.iloc[tr_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        # CatBoost
        cb = CatBoostClassifier(**cat_params)
        cb.fit(Pool(X_tr, y_tr, cat_features=cat_feature_indices),
               eval_set=Pool(X_val, y_val, cat_features=cat_feature_indices))
        oof_cat[val_idx]  = cb.predict_proba(Pool(X_val, cat_features=cat_feature_indices))[:, 1]
        test_cat         += cb.predict_proba(Pool(X_test, cat_features=cat_feature_indices))[:, 1] / N_SPLITS

        # LightGBM
        lb = lgb.LGBMClassifier(**lgb_params)
        lb.fit(X_lgb.iloc[tr_idx], y_tr,
               eval_set=[(X_lgb.iloc[val_idx], y_val)],
               callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_lgb[val_idx]  = lb.predict_proba(X_lgb.iloc[val_idx])[:, 1]
        test_lgb         += lb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

        # XGBoost
        xb = xgb.XGBClassifier(**xgb_params)
        xb.fit(X_xgb.iloc[tr_idx], y_tr,
               eval_set=[(X_xgb.iloc[val_idx], y_val)],
               verbose=False)
        oof_xgb[val_idx]  = xb.predict_proba(X_xgb.iloc[val_idx])[:, 1]
        test_xgb         += xb.predict_proba(X_test_xgb)[:, 1] / N_SPLITS

    # 이 SEED의 최적 가중치 탐색
    best_auc, best_w = 0.0, (0.6, 0.15, 0.25)
    for w1 in np.arange(0.0, 1.05, 0.05):
        for w2 in np.arange(0.0, 1.05 - w1, 0.05):
            w3 = round(1 - w1 - w2, 2)
            if w3 < 0: continue
            auc = roc_auc_score(y, oof_cat * w1 + oof_lgb * w2 + oof_xgb * w3)
            if auc > best_auc:
                best_auc, best_w = auc, (w1, w2, w3)

    print(f'SEED {SEED} | CAT: {roc_auc_score(y, oof_cat):.5f} | '
          f'LGB: {roc_auc_score(y, oof_lgb):.5f} | '
          f'XGB: {roc_auc_score(y, oof_xgb):.5f} | '
          f'앙상블: {best_auc:.5f} (w={best_w[0]:.2f},{best_w[1]:.2f},{best_w[2]:.2f})')

    seed_results.append({'seed': SEED, 'auc': best_auc, 'w': best_w})

    # 전체 누적
    oof_cat_all  += oof_cat  / len(SEEDS)
    oof_lgb_all  += oof_lgb  / len(SEEDS)
    oof_xgb_all  += oof_xgb  / len(SEEDS)
    test_cat_all += test_cat / len(SEEDS)
    test_lgb_all += test_lgb / len(SEEDS)
    test_xgb_all += test_xgb / len(SEEDS)

print('\n모든 SEED 학습 완료')

SyntaxError: f-string: expecting '}' (1357790322.py, line 16)

In [ ]:
# Seed Averaging 최종 앙상블
# SEED별 결과 출력
print('=== SEED별 앙상블 AUC ===')
for r in seed_results:
    print(f"  SEED {r['seed']:4d} | AUC: {r['auc']:.5f}")
print(f"  평균: {np.mean([r['auc'] for r in seed_results]):.5f}")
print(f"  편차: {np.std([r['auc'] for r in seed_results]):.5f}")
print()

# 평균된 OOF로 최적 가중치 재탐색
best_auc_avg, best_w_avg = 0.0, (0.6, 0.15, 0.25)
results = []

for w1 in np.arange(0.0, 1.05, 0.05):
    for w2 in np.arange(0.0, 1.05 - w1, 0.05):
        w3 = round(1 - w1 - w2, 2)
        if w3 < 0: continue
        blended = oof_cat_all * w1 + oof_lgb_all * w2 + oof_xgb_all * w3
        auc = roc_auc_score(y, blended)
        results.append((round(w1,2), round(w2,2), round(w3,2), auc))
        if auc > best_auc_avg:
            best_auc_avg = auc
            best_w_avg = (w1, w2, w3)

results_df = pd.DataFrame(results, columns=['w_cat','w_lgb','w_xgb','oof_auc'])
print('=== Seed Averaging 후 최적 가중치 Top 10 ===')
display(results_df.sort_values('oof_auc', ascending=False).head(10))

print(f'\n✅ Seed Averaging OOF AUC : {best_auc_avg:.5f}')
print(f'   최적 가중치              : CAT={best_w_avg[0]:.2f}, LGB={best_w_avg[1]:.2f}, XGB={best_w_avg[2]:.2f}')
print(f'   이전 최고 (OOF)          : 0.74046')
print(f'   이전 최고 (리더보드)     : 0.74193')
print(f'   OOF 개선폭               : {best_auc_avg - 0.74046:+.5f}')

In [ ]:
# 제출 파일 생성
test_blended = (test_cat_all * best_w_avg[0] +
                test_lgb_all * best_w_avg[1] +
                test_xgb_all * best_w_avg[2])

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_blended
submission.to_csv('submission_seed_avg.csv', index=False, encoding='utf-8-sig')

print('저장 완료: submission_seed_avg.csv')
print(f'\n최종 요약')
print(f'  사용 SEED          : {SEEDS}')
print(f'  Seed Avg OOF AUC  : {best_auc_avg:.5f}')
print(f'  이전 최고 (OOF)   : 0.74046')
print(f'  이전 최고 (LB)    : 0.74193')
print(f'  OOF 개선폭        : {best_auc_avg - 0.74046:+.5f}')
display(submission.head())